# API Integration with LangChain using Gemini API

In [ ]:
import os,warnings
from dotenv import load_dotenv
import json
import requests
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# --------------------------------------------
# 1) Create an API integration tool
# --------------------------------------------

@tool
def get_article_by_id(article_id: int) -> str:
    """
    Retrieve an article from the external content API using its ID.

    Use this tool when the user asks to retrieve, explain or summarize a particular article.
    """

    if article_id < 1 or article_id > 100:
        return json.dumps({ "error": "Article ID must be between 1 and 100."})

    url = f"https://jsonplaceholder.typicode.com/posts/{article_id}"

    try:
        response = requests.get(url,timeout=10)
        response.raise_for_status()

        article = response.json()

        if not article:
            return json.dumps({"error": f"Article {article_id} was not found." })

        return json.dumps({
            "article_id": article["id"],
            "author_id": article["userId"],
            "title": article["title"],
            "content": article["body"]
        })

    except requests.Timeout:
        return json.dumps({"error": "The external API request timed out." })

    except requests.RequestException as e:
        return json.dumps({"error": f"Unable to access the external API: {str(e)}" })

In [ ]:
# get_article_by_id(45)

In [14]:
# 2) Create the Gemini LLM

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key,temperature=0)


# 3) Register the API tool
tools = [get_article_by_id]


# 4) Create the LangChain agent

agent = create_agent(model=llm, tools=tools,
    system_prompt="""
    You are a content assistant.

    When a user asks about an article:
    1. Use the available API tool to retrieve the article.
    2. Do not invent article details.
    3. If the API returns an error, explain it clearly.
    4. Otherwise, provide the article title and a concise summary.
    """
)

article_id = 77

# 5) Prepare the user request

user_query = f"""
Retrieve article number {article_id}.

Give me:
1. The article title
2. A two-sentence summary
"""

input_data = {"messages": [{"role": "user","content": user_query}]}

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [15]:
# 6) Run the agent
response = agent.invoke(input_data)

In [16]:
# 7) Print the final response

final_message = response["messages"][-1]
print(final_message.text)

**Title:** necessitatibus quasi exercitationem odio

**Summary:** This article explores themes of human experience and the inevitability of various life circumstances. It reflects on the nature of change and the complex emotions that arise from navigating these universal challenges.
